## 0. 환경설정 — 스드메 통합 챗봇

하나의 채팅방에서 스튜디오/드레스/메이크업 질문을 자유롭게 할 수 있습니다.

In [60]:
!pip install neo4j-graphrag neo4j openai mysql-connector-python gradio python-dotenv -q

In [61]:
import os, json, re, time
from dotenv import load_dotenv

load_dotenv()

from neo4j import GraphDatabase, basic_auth
from neo4j_graphrag.retrievers import Text2CypherRetriever
try:
    from neo4j_graphrag.llm import OpenAILLM
except ImportError:
    from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG, RagTemplate
from openai import OpenAI
import gradio as gr

print("모듈 로드 완료")

모듈 로드 완료


In [62]:
client = OpenAI()

## 1. Neo4j 연결 (읽기 전용 — 데이터 적재는 db_load.py)

In [63]:
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://127.0.0.1:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ["NEO4J_PW"]

driver = GraphDatabase.driver(NEO4J_URI, auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    cnt = session.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
    print(f"Neo4j 현재 노드 수: {cnt}")

Neo4j 현재 노드 수: 15959


In [64]:
# 카테고리별 데이터 존재 확인
with driver.session() as session:
    for cat in ["studio", "dress", "makeup"]:
        cnt = session.run(
            "MATCH (v:Vendor {category: $cat}) RETURN count(v) AS cnt", cat=cat
        ).single()["cnt"]
        status = f"{cnt}개 확인" if cnt > 0 else "⚠ 0개 — db_load.py 실행 필요"
        print(f"  {cat}: {status}")

  studio: 207개 확인
  dress: 268개 확인
  makeup: 472개 확인


## 2. MySQL (더미 fallback)

In [65]:
import mysql.connector

MYSQL_HOST = os.environ.get("MYSQL_HOST", "")
MYSQL_USER = os.environ.get("MYSQL_USER", "")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "")
MYSQL_DB = os.environ.get("MYSQL_DB", "")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT", 3306))
COUPLE_ID = 2

DUMMY_PREF = {"region": "서울", "sub_region": "강남구",
              "studio_style": "인물중심", "dress_style": "클래식", "makeup_style": "내추럴"}
DUMMY_LIKES = [{"name": "더미업체", "category": "스튜디오"}]

conn = None
if all([MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB]):
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST, user=MYSQL_USER,
            password=MYSQL_PASSWORD, database=MYSQL_DB,
            port=MYSQL_PORT
        )
        print("MySQL 연결 성공")
    except Exception as e:
        conn = None
        print(f"MySQL 연결 실패: {e}")
else:
    print("MySQL 미설정 - 더미 사용")

def get_user_preference(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_preferences WHERE couple_id = %s", (couple_id,))
            row = cur.fetchone(); cur.close()
            if row:
                return row
        except:
            pass
    return DUMMY_PREF

def get_user_likes(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_venue_likes WHERE couple_id = %s", (couple_id,))
            rows = cur.fetchall(); cur.close()
            if rows:
                return rows
        except:
            pass
    return DUMMY_LIKES

print("사용자 함수 정의 완료")

MySQL 연결 성공
사용자 함수 정의 완료


## 3. Neo4j 스키마 추출

In [66]:
def get_schema(uri, user, password):
    d = GraphDatabase.driver(uri, auth=basic_auth(user, password))
    with d.session() as session:
        nodes = session.run("MATCH (n) WITH DISTINCT labels(n) AS lbls, keys(n) AS ks, n UNWIND lbls AS l UNWIND ks AS k RETURN l, k, n[k] AS sv")
        rels = session.run("MATCH (a)-[r]->(b) RETURN DISTINCT labels(a) AS sl, type(r) AS rt, labels(b) AS el")
        schema = {"nodes": {}, "relations": []}
        for rec in nodes:
            l, k = rec["l"], rec["k"]
            if l not in schema["nodes"]: schema["nodes"][l] = {}
            v = rec["sv"]
            t = "STRING" if isinstance(v, str) else "INTEGER" if isinstance(v, int) else "FLOAT" if isinstance(v, float) else "UNKNOWN"
            schema["nodes"][l][k] = t
        for rec in rels:
            sl = rec["sl"][0] if rec["sl"] else "?"
            el = rec["el"][0] if rec["el"] else "?"
            schema["relations"].append(f"(:{sl})-[:{rec['rt']}]->(:{el})")
    d.close()
    return schema

def format_schema(schema):
    lines = ["Node properties:"]
    for label, props in schema["nodes"].items():
        ps = ", ".join(f"{k}: {v}" for k, v in props.items())
        lines.append(f"  {label} {{{ps}}}")
    lines.append("Relationships:")
    for r in schema["relations"]: lines.append(f"  {r}")
    return "\n".join(lines)

schema = get_schema(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
neo4j_schema = format_schema(schema)
print(neo4j_schema)

Node properties:
  Vendor {embedding: UNKNOWN, likeCnt: INTEGER, iweddingNo: STRING, productPrice: INTEGER, category: STRING, profile: STRING, eventOptionPrice: INTEGER, orderCnt: INTEGER, detailCmt: STRING, enterpriseCode: STRING, salePrice: INTEGER, holiday: STRING, viewCnt: INTEGER, eventPrice: INTEGER, subCategory: STRING, reviewCnt: INTEGER, profileUrl: STRING, rating: FLOAT, productName: STRING, region: STRING, coverUrl: STRING, typeName: STRING, tel: STRING, partnerId: INTEGER, name: STRING, address: STRING, uuid: STRING}
  Region {name: STRING}
  Tag {category: STRING, name: STRING, typeName: STRING, type: INTEGER}
  StyleFilter {name: STRING, partnerType: INTEGER, id: INTEGER}
  Review {name: STRING, date: STRING, score: INTEGER, contents: STRING}
  Package {desc: STRING, title: STRING, value: STRING}
  Hall {minRentalPrice: INTEGER, rating: FLOAT, reviewCnt: INTEGER, availableContract: INTEGER, profileUrl: STRING, address: STRING, maxIndividualHallPrice: INTEGER, tel: STRING,

## 4. Few-shot Cypher 예시 (3개 카테고리 통합)

스튜디오/드레스/메이크업 예시가 모두 포함되어 있어, GraphRAG가 질문에서 카테고리를 자동 판별합니다.

In [67]:
examples = [
    # ============================================
    # 스튜디오 (category: 'studio')
    # ============================================
    """USER INPUT: '스튜디오 추천해줘'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '200만원 이하 스튜디오'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.salePrice <= 2000000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '야외씬 잘 찍는곳'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '야외' OR t.name CONTAINS '로드' OR t.name CONTAINS '가든'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '인물중심 스타일 추천'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '인물중심'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '강남 스튜디오 150만원 이하'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남' AND v.salePrice <= 1500000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '역삼역 근처 스튜디오'
QUERY:
MATCH (v:Vendor {category:'studio'})
WHERE v.address CONTAINS '역삼'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '논현동 메이크업샵'
QUERY:
MATCH (v:Vendor {category:'makeup'})
WHERE v.address CONTAINS '논현'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '신사동 드레스샵 추천'
QUERY:
MATCH (v:Vendor {category:'dress'})
WHERE v.address CONTAINS '신사'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '줄리의정원 가격이랑 패키지 알려줘'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.name CONTAINS '줄리의정원'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
RETURN v.name AS name, v.salePrice AS salePrice, v.productPrice AS originalPrice, v.rating AS rating, v.address AS address,
       collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages""",

    """USER INPUT: '줄리의정원이랑 식스플로어 비교해줘'
QUERY:
MATCH (v:Vendor {category:'studio'})
WHERE v.name CONTAINS '줄리의정원' OR v.name CONTAINS '식스플로어'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, collect(DISTINCT t.name) AS tags, avg(rv.score) AS avgScore, count(rv) AS revCnt
RETURN v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, tags, round(avgScore, 1) AS avgScore, revCnt
ORDER BY v.name""",

    """USER INPUT: '줄리의정원과 비슷한 스타일'
QUERY:
MATCH (v1:Vendor {category:'studio'})-[:HAS_TAG]->(t:Tag)<-[:HAS_TAG]-(v2:Vendor {category:'studio'})
WHERE v1.name CONTAINS '줄리의정원' AND v1 <> v2
WITH v2, collect(DISTINCT t.name) AS sharedTags, count(t) AS cnt
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, v2.address AS address, sharedTags
ORDER BY cnt DESC, v2.rating DESC LIMIT 5""",

    """USER INPUT: '야외씬이랑 어울리는 스타일은?'
QUERY:
MATCH (t1:Tag {category:'studio'})-[c:CO_OCCURS]-(t2:Tag {category:'studio'})
WHERE t1.name CONTAINS '야외'
RETURN t2.name AS relatedTag, c.count AS freq
ORDER BY freq DESC LIMIT 10""",

    """USER INPUT: '리뷰 좋은 강남 스튜디오'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남'
MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, avg(rv.score) AS avgScore, count(rv) AS revCnt
WHERE revCnt >= 3
RETURN v.name AS name, v.salePrice AS price, v.address AS address, round(avgScore, 1) AS avgScore, revCnt
ORDER BY avgScore DESC LIMIT 10""",

    # ============================================
    # 드레스 (category: 'dress')
    # ============================================
    """USER INPUT: '드레스 추천해줘'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '200만원 이하 드레스'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.salePrice <= 2000000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '촬영+본식 드레스 4벌 이상'
QUERY:
MATCH (v:Vendor {category:'dress'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '4벌' OR t.name CONTAINS '5벌'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '컬러 드레스 있는 곳'
QUERY:
MATCH (v:Vendor {category:'dress'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '컬러'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '셀렉션H 가격이랑 구성 알려줘'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.name CONTAINS '셀렉션'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
RETURN v.name AS name, v.salePrice AS salePrice, v.productPrice AS originalPrice, v.rating AS rating, v.address AS address,
       collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages""",

    """USER INPUT: '셀렉션H와 비슷한 스타일 드레스샵'
QUERY:
MATCH (v1:Vendor {category:'dress'})-[:HAS_TAG]->(t:Tag)<-[:HAS_TAG]-(v2:Vendor {category:'dress'})
WHERE v1.name CONTAINS '셀렉션' AND v1 <> v2
WITH v2, collect(DISTINCT t.name) AS sharedTags, count(t) AS cnt
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, v2.address AS address, sharedTags
ORDER BY cnt DESC, v2.rating DESC LIMIT 5""",

    """USER INPUT: '리뷰 좋은 강남 드레스샵'
QUERY:
MATCH (v:Vendor {category:'dress'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남'
MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, avg(rv.score) AS avgScore, count(rv) AS revCnt
WHERE revCnt >= 3
RETURN v.name AS name, v.salePrice AS price, v.address AS address, round(avgScore, 1) AS avgScore, revCnt
ORDER BY avgScore DESC LIMIT 10""",

    # ============================================
    # 메이크업 (category: 'makeup')
    # ============================================
    """USER INPUT: '메이크업 추천해줘'
QUERY:
MATCH (v:Vendor {category:'makeup'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '50만원 이하 메이크업'
QUERY:
MATCH (v:Vendor {category:'makeup'}) WHERE v.salePrice <= 500000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '내추럴 메이크업 추천'
QUERY:
MATCH (v:Vendor {category:'makeup'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '내추럴' OR t.name CONTAINS '깨끗'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '실장급 메이크업'
QUERY:
MATCH (v:Vendor {category:'makeup'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '실장'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '르보청담 가격이랑 구성 알려줘'
QUERY:
MATCH (v:Vendor {category:'makeup'}) WHERE v.name CONTAINS '르보'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
RETURN v.name AS name, v.salePrice AS salePrice, v.productPrice AS originalPrice, v.rating AS rating, v.address AS address,
       collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages""",

    """USER INPUT: '르보청담과 비슷한 스타일 메이크업샵'
QUERY:
MATCH (v1:Vendor {category:'makeup'})-[:HAS_TAG]->(t:Tag)<-[:HAS_TAG]-(v2:Vendor {category:'makeup'})
WHERE v1.name CONTAINS '르보' AND v1 <> v2
WITH v2, collect(DISTINCT t.name) AS sharedTags, count(t) AS cnt
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, v2.address AS address, sharedTags
ORDER BY cnt DESC, v2.rating DESC LIMIT 5""",

    """USER INPUT: '리뷰 좋은 강남 메이크업샵'
QUERY:
MATCH (v:Vendor {category:'makeup'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남'
MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, avg(rv.score) AS avgScore, count(rv) AS revCnt
WHERE revCnt >= 3
RETURN v.name AS name, v.salePrice AS price, v.address AS address, round(avgScore, 1) AS avgScore, revCnt
ORDER BY avgScore DESC LIMIT 10""",

    # ============================================
    # 공통 패턴
    # ============================================
    """USER INPUT: '요즘 인기있는 곳'
QUERY:
MATCH (v:Vendor) WHERE v.orderCnt > 0
RETURN v.partnerId AS id, v.name AS name, v.category AS category, v.salePrice AS price, v.rating AS rating, v.address AS address, v.orderCnt AS orders, v.profileUrl AS url
ORDER BY v.orderCnt DESC LIMIT 10""",

    """USER INPUT: '가장 저렴한 곳 5개'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.category AS category, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.salePrice ASC LIMIT 5""",

    # ============================================
    # 복합 질문 (UNION 사용 금지 — 하나의 MATCH로 처리)
    # ============================================
    """USER INPUT: '로이스튜디오는 어디에 있고 근처에 다른 스튜디오도 추천해줘'
QUERY:
MATCH (v1:Vendor {category:'studio'})-[:IN_REGION]->(r:Region)<-[:IN_REGION]-(v2:Vendor {category:'studio'})
WHERE v1.name CONTAINS '로이스튜디오'
RETURN v1.name AS name, v1.address AS address, v1.salePrice AS price, v1.rating AS rating, r.name AS region,
       collect(DISTINCT {name: v2.name, price: v2.salePrice, rating: v2.rating})[..5] AS nearbyVendors""",

    """USER INPUT: '줄리의정원 정보랑 비슷한 가격대 다른 곳도 알려줘'
QUERY:
MATCH (v1:Vendor {category:'studio'})-[:IN_REGION]->(r:Region)<-[:IN_REGION]-(v2:Vendor {category:'studio'})
WHERE v1.name CONTAINS '줄리의정원' AND v1 <> v2 AND abs(v2.salePrice - v1.salePrice) < 300000
RETURN v1.name AS name, v1.address AS address, v1.salePrice AS price, v1.rating AS rating, r.name AS region,
       collect(DISTINCT {name: v2.name, price: v2.salePrice, rating: v2.rating})[..5] AS similarPriceVendors""",

    # ============================================
    # 이전 대화 맥락 참조 패턴
    # ============================================
    """USER INPUT: '아까 추천한 업체 중에서 가장 싼 곳'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.category AS category, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.salePrice ASC LIMIT 5""",

    """USER INPUT: '그 업체 상세 정보 알려줘'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS $vendorName
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
OPTIONAL MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages, collect(DISTINCT {name: rv.name, score: rv.score, contents: rv.contents})[..5] AS reviews
RETURN v.name AS name, v.category AS category, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.address AS address, v.tel AS tel, tags, packages, reviews""",
]

print(f"Few-shot 예시 {len(examples)}개 로드 완료 (스튜디오/드레스/메이크업 통합)")

Few-shot 예시 33개 로드 완료 (스튜디오/드레스/메이크업 통합)


In [ ]:
# 동적 LIMIT 예시 추가 (사용자가 개수를 지정할 때 LIMIT을 맞추도록)
examples.extend([
    """USER INPUT: '스튜디오 15개 추천해줘'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 15""",

    """USER INPUT: '드레스 20개 보여줘'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 20""",
])

print(f"Few-shot 예시 {len(examples)}개 (동적 LIMIT 포함)")

## 5. LLM / Retriever / GraphRAG / Tool 정의

In [ ]:
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.types import RetrieverResultItem

SDM_RAG_TEMPLATE = RagTemplate(
    template="""당신은 웨딩 스드메(스튜디오/드레스/메이크업) 전문 추천 챗봇입니다.

아래 검색 결과를 기반으로 답변하세요.

[답변 규칙]
1. 검색 결과에 있는 데이터만 사용. 없는 내용을 만들지 마세요.
2. 추천 시 업체명, 가격, 평점, 특징을 포함해 구체적으로 답변.
3. 가격은 만원 단위로 표시 (예: 163만원).
4. 여러 업체를 추천할 때는 번호를 매겨 정리.
5. 비교 질문에는 항목별로 나눠서 비교.
6. 동일 업체가 가격만 다르게 여러 개 있으면 하나로 합쳐서 가격 범위로 표시.
7. 데이터가 없으면 "해당 조건의 업체를 찾지 못했습니다"라고 솔직하게 답변.

검색 결과:
{context}

사용자 질문: {query_text}

답변:""",
    expected_inputs=["context", "query_text"]
)

llm = OpenAILLM(model_name="gpt-4o", model_params={"temperature": 0, "max_tokens": 2000})
text2cypher_retriever = Text2CypherRetriever(driver=driver, llm=llm, neo4j_schema=neo4j_schema, examples=examples)
rag_cypher = GraphRAG(retriever=text2cypher_retriever, llm=llm, prompt_template=SDM_RAG_TEMPLATE)

embedder = OpenAIEmbeddings(model="text-embedding-3-small")

NO_RESULT_PHRASES = ["찾지 못했습니다", "없습니다", "검색 결과가 없"]


def _vendor_result_formatter(record):
    return RetrieverResultItem(
        content=json.dumps(dict(record), ensure_ascii=False, default=str),
        metadata={"name": record.get("name", "")}
    )


def build_vector_retrieval_query(category=None, region=None, max_price=None, min_price=None):
    parts = ["WITH node AS v, score"]
    if category:
        parts.append(f"WHERE v.category = '{category}'")
    parts.append("MATCH (v)-[:IN_REGION]->(r:Region)")
    parts.append("OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)")
    parts.append("OPTIONAL MATCH (v)-[:HAS_REVIEW]->(rv:Review)")
    parts.append("""WITH v, score, r,
        collect(DISTINCT t.name) AS tags,
        avg(rv.score) AS avgReviewScore,
        count(rv) AS reviewCount""")
    score_parts = []
    if region:
        score_parts.append(f"CASE WHEN r.name CONTAINS '{region}' THEN 1 ELSE 0 END AS regionMatch")
    if max_price:
        score_parts.append(f"CASE WHEN v.salePrice <= {max_price} AND v.salePrice > 0 THEN 1 ELSE 0 END AS priceMatch")
    if min_price:
        score_parts.append(f"CASE WHEN v.salePrice >= {min_price} THEN 1 ELSE 0 END AS priceMinMatch")
    if score_parts:
        parts.append("WITH v, score, r, tags, avgReviewScore, reviewCount,")
        parts.append(",\n    ".join(score_parts))
        score_names = [s.split(" AS ")[1] for s in score_parts]
        condition_score = " + ".join(score_names)
        parts.append(f"""RETURN v.name AS name, v.category AS category,
    v.salePrice AS price, v.rating AS rating,
    v.address AS address, v.profileUrl AS url,
    tags, round(avgReviewScore, 1) AS avgReviewScore,
    reviewCount, round(score, 4) AS vectorScore,
    {', '.join(score_names)},
    ({condition_score}) AS conditionScore
ORDER BY conditionScore DESC, score DESC LIMIT 10""")
    else:
        parts.append(f"""RETURN v.name AS name, v.category AS category,
    v.salePrice AS price, v.rating AS rating,
    v.address AS address, v.profileUrl AS url,
    tags, round(avgReviewScore, 1) AS avgReviewScore,
    reviewCount, round(score, 4) AS vectorScore
ORDER BY score DESC LIMIT 10""")
    return "\n".join(parts)


def create_vector_rag(category=None, region=None, max_price=None, min_price=None):
    rq = build_vector_retrieval_query(category, region, max_price, min_price)
    vr = VectorCypherRetriever(
        driver=driver, index_name="vendor_embedding_index",
        embedder=embedder, retrieval_query=rq,
        result_formatter=_vendor_result_formatter,
    )
    return GraphRAG(retriever=vr, llm=llm, prompt_template=SDM_RAG_TEMPLATE), rq


# ── session_state ──
session_state = {
    "category": None,
    "vendors": [],
    "last_mentioned": [],
    "vendor_history": {},
    "turn": 0,
}


def extract_vendors_from_retriever(result):
    vendors = []
    if hasattr(result, "retriever_result") and result.retriever_result:
        for item in getattr(result.retriever_result, "items", []):
            name = None
            if item.metadata and isinstance(item.metadata, dict):
                name = item.metadata.get("name")
            if not name and item.content and isinstance(item.content, str):
                m = re.search(r'"name":\s*"([^"]+)"', item.content)
                if m:
                    name = m.group(1)
                if not name:
                    m = re.search(r"name='([^']+)'", item.content)
                    if m:
                        name = m.group(1)
                if not name:
                    m = re.search(r"name[=:]\s*['\"]?([^'\",\n\}]+)", item.content)
                    if m:
                        name = m.group(1).strip()
            if name and 2 <= len(name) <= 30:
                if name not in vendors:
                    vendors.append(name)
    return vendors


def extract_vendors_from_answer(answer):
    vendors = []
    lines = answer.split("\n")
    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue
        if i + 1 < len(lines):
            next_line = lines[i + 1].strip()
            if next_line.startswith("가격:") or next_line.startswith("가격 :"):
                name = re.sub(r"^\d+[\.\)]\s*", "", line)
                name = name.strip("*").strip()
                if 2 <= len(name) <= 30 and name not in vendors:
                    vendors.append(name)
    return vendors


def query_vendors_by_names(vendor_names):
    with driver.session() as session:
        records = session.run("""
            MATCH (v:Vendor)
            WHERE any(name IN $names WHERE v.name = name)
            WITH v
            OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
            WITH v, collect(DISTINCT t.name) AS tags
            OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
            WITH v, tags, collect(DISTINCT {title: p.title, value: p.value})[..3] AS packages
            OPTIONAL MATCH (v)-[:HAS_REVIEW]->(rv:Review)
            WITH v, tags, packages,
                 round(avg(rv.score), 1) AS avgReviewScore,
                 count(rv) AS reviewCount,
                 collect(rv.contents)[..2] AS recentReviews
            RETURN v.name AS name, v.category AS category,
                   v.salePrice AS price, v.rating AS rating,
                   v.address AS address, v.region AS region,
                   v.profileUrl AS url, v.holiday AS holiday,
                   v.reviewCnt AS reviewCnt,
                   tags, packages, avgReviewScore,
                   reviewCount, recentReviews
            ORDER BY v.rating DESC
        """, names=vendor_names).data()
    return records


# ── Tool 함수 ──

def tool_search_structured(query, category, **kwargs):
    result = rag_cypher.search(query_text=query)
    vendors = extract_vendors_from_retriever(result)
    answer = result.answer
    if not vendors:
        vendors = extract_vendors_from_answer(answer)
    if answer and any(p in answer for p in NO_RESULT_PHRASES):
        vec_rag, _ = create_vector_rag(category=category)
        result = vec_rag.search(query_text=query)
        vendors = extract_vendors_from_retriever(result)
        answer = result.answer
    return "graphrag", answer, vendors


def tool_search_semantic(query, category, region=None, max_price=None, min_price=None, **kwargs):
    vec_rag, _ = create_vector_rag(category=category, region=region,
                                    max_price=max_price, min_price=min_price)
    result = vec_rag.search(query_text=query)
    vendors = extract_vendors_from_retriever(result)
    return "graphrag", result.answer, vendors


def tool_compare_vendors(vendor_names, criteria=None, **kwargs):
    records = query_vendors_by_names(vendor_names)
    data = json.dumps(records, ensure_ascii=False, default=str)
    vendor_list = list(dict.fromkeys(r["name"] for r in records))
    return "raw", data, vendor_list


def tool_filter_previous(vendor_names, condition, count=None, **kwargs):
    records = query_vendors_by_names(vendor_names)
    data = json.dumps(records, ensure_ascii=False, default=str)
    vendor_list = list(dict.fromkeys(r["name"] for r in records))
    return "raw", data, vendor_list


def tool_get_vendor_detail(vendor_name, **kwargs):
    records = query_vendors_by_names([vendor_name])
    data = json.dumps(records, ensure_ascii=False, default=str)
    return "raw", data, [vendor_name]


def tool_get_user_preference(**kwargs):
    pref = get_user_preference(conn, COUPLE_ID)
    lines = [f"- {k}: {v}" for k, v in pref.items() if k != "couple_id" and v]
    return "direct", "현재 저장된 취향 정보입니다.\n" + "\n".join(lines), []


def tool_get_user_likes(**kwargs):
    likes = get_user_likes(conn, COUPLE_ID)
    if likes:
        lines = [f"- {l.get('name','알수없음')} ({l.get('category','')})" for l in likes]
        return "direct", f"좋아요한 업체가 {len(likes)}건 있습니다.\n" + "\n".join(lines), []
    return "direct", "좋아요한 업체가 없습니다.", []


TOOL_MAP = {
    "search_structured": tool_search_structured,
    "search_semantic": tool_search_semantic,
    "compare_vendors": tool_compare_vendors,
    "filter_previous": tool_filter_previous,
    "get_vendor_detail": tool_get_vendor_detail,
    "get_user_preference": tool_get_user_preference,
    "get_user_likes": tool_get_user_likes,
}

TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "search_structured",
        "description": "구조적 조건으로 웨딩 업체 검색. 가격, 지역, 태그, 정렬 조건이 있을 때 사용. 조건이 부족해도 되묻지 말고 있는 조건으로 검색.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "사용자 원문 그대로 전달. 축약 금지."},
            "category": {"type": "string", "enum": ["studio", "dress", "makeup"]},
        }, "required": ["query", "category"]}
    }},
    {"type": "function", "function": {
        "name": "search_semantic",
        "description": "스타일/분위기/느낌 등 추상적 표현으로 검색. 의미 유사도 매칭. 가격/지역 병행 가능.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "사용자 원문 그대로 전달. 축약 금지."},
            "category": {"type": "string", "enum": ["studio", "dress", "makeup"]},
            "region": {"type": "string", "description": "지역명 (선택)"},
            "max_price": {"type": "integer", "description": "최대 가격 원 단위 (선택)"},
            "min_price": {"type": "integer", "description": "최소 가격 원 단위 (선택)"},
        }, "required": ["query", "category"]}
    }},
    {"type": "function", "function": {
        "name": "compare_vendors",
        "description": "이전에 추천된 업체들을 비교. 대화 맥락에서 업체명을 파악하여 호출.",
        "parameters": {"type": "object", "properties": {
            "vendor_names": {"type": "array", "items": {"type": "string"}},
            "criteria": {"type": "string", "description": "비교 기준"},
        }, "required": ["vendor_names"]}
    }},
    {"type": "function", "function": {
        "name": "filter_previous",
        "description": "이전 추천 결과에서 필터링/정렬. '이중에서', '평점순', '가까운 곳' 등에 사용.",
        "parameters": {"type": "object", "properties": {
            "vendor_names": {"type": "array", "items": {"type": "string"}},
            "condition": {"type": "string"},
            "count": {"type": "integer"},
        }, "required": ["vendor_names", "condition"]}
    }},
    {"type": "function", "function": {
        "name": "get_vendor_detail",
        "description": "특정 업체 상세 정보 조회.",
        "parameters": {"type": "object", "properties": {
            "vendor_name": {"type": "string"},
        }, "required": ["vendor_name"]}
    }},
    {"type": "function", "function": {
        "name": "get_user_preference",
        "description": "사용자 취향/선호 정보 조회.",
        "parameters": {"type": "object", "properties": {}}
    }},
    {"type": "function", "function": {
        "name": "get_user_likes",
        "description": "사용자 좋아요/찜 목록 조회.",
        "parameters": {"type": "object", "properties": {}}
    }},
]

SYSTEM_PROMPT = """당신은 웨딩 스드메(스튜디오/드레스/메이크업) 전문 추천 챗봇입니다.
서울/경기 지역의 웨딩 스튜디오, 드레스, 메이크업 업체를 추천합니다.

[핵심 원칙]
- 웨딩 스드메 관련 질문에는 반드시 tool을 호출하세요. 되묻지 마세요.
- 조건이 부족해도 있는 조건으로 일단 검색하세요. "어떤 지역?" "예산은?" 되묻기 금지.
- 카테고리를 모르면 [현재 대화 상태]에서 확인하세요.
- 웨딩 스드메와 무관한 질문에만 tool 없이 직접 답변하세요.

[주의사항]
- "여기서", "이중에", "그거", "아까" 등 맥락 참조 시 [현재 대화 상태]의 업체명을 사용하세요.
- "촬영"이 카테고리인지 조건인지 문맥에서 판단하세요.
- "100만원에 가까운" = 약 70만~130만원 범위.
- 동일 업체가 가격만 다르면 하나로 합쳐서 가격 범위로 표시.
"""

print("GraphRAG + Tool 정의 완료")
print(f"  - Tool 7개: {list(TOOL_MAP.keys())}")

## 6. Gradio 챗봇

In [ ]:
def build_dynamic_system_prompt():
    """session_state를 포함한 동적 시스템 프롬프트 생성"""
    state_lines = []
    cat = session_state.get("category")
    if cat:
        cat_kr = {"studio": "스튜디오", "dress": "드레스", "makeup": "메이크업"}.get(cat, cat)
        state_lines.append(f"현재 카테고리: {cat_kr}")
    vendors = session_state.get("vendors", [])
    if vendors:
        state_lines.append(f"추천 업체 목록: {', '.join(vendors[:10])}")
    last = session_state.get("last_mentioned", [])
    if last and last != vendors:
        state_lines.append(f"직전 언급 업체: {', '.join(last[:5])}")
    history = session_state.get("vendor_history", {})
    for hcat, hvendors in history.items():
        if hcat != cat and hvendors:
            hcat_kr = {"studio": "스튜디오", "dress": "드레스", "makeup": "메이크업"}.get(hcat, hcat)
            state_lines.append(f"이전 {hcat_kr} 추천: {', '.join(hvendors[:5])}")

    state_block = ""
    if state_lines:
        state_block = "\n[현재 대화 상태]\n" + "\n".join(state_lines)

    return SYSTEM_PROMPT + state_block


def response_fn(message, chat_history, debug_log):
    chat_history = chat_history or []
    msg = message.strip()
    log_lines = [f"[입력] {msg}"]
    log_lines.append(f"[history] {len(chat_history)}개 메시지, GPT에 {min(len(chat_history), 6)}개 전달")

    if session_state["vendors"]:
        log_lines.append(f"[state] cat={session_state['category']}, "
                         f"vendors={session_state['vendors'][:3]}, "
                         f"last={session_state['last_mentioned'][:2]}, "
                         f"turn={session_state['turn']}")

    # ── 1단계: GPT-4o tool 선택 ──
    dynamic_prompt = build_dynamic_system_prompt()
    messages = [{"role": "system", "content": dynamic_prompt}]
    for m in chat_history[-6:]:
        messages.append({"role": m["role"], "content": m["content"]})
    messages.append({"role": "user", "content": msg})

    try:
        t0 = time.time()
        response = client.chat.completions.create(
            model="gpt-4o", messages=messages,
            tools=TOOLS_SCHEMA, tool_choice="auto", temperature=0,
        )
        tool_select_time = time.time() - t0
        if response.usage:
            log_lines.append(f"[tokens] prompt={response.usage.prompt_tokens}, "
                             f"completion={response.usage.completion_tokens}")
    except Exception as e:
        log_lines.append(f"[오류] Tool 선택 실패: {e}")
        answer = "죄송합니다. 처리 중 오류가 발생했습니다."
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": answer})
        new_log = "\n".join(log_lines)
        full_log = f"{debug_log}\n{'─'*40}\n{new_log}" if debug_log else new_log
        return "", chat_history, full_log

    choice = response.choices[0]
    all_vendors = []
    is_new_search = False

    # ── 2단계: Tool 실행 ──
    if choice.message.tool_calls:
        num_calls = len(choice.message.tool_calls)
        log_lines.append(f"[tool_calls] {num_calls}개 ({tool_select_time:.1f}초)")

        tool_results = []
        for tc in choice.message.tool_calls:
            tool_name = tc.function.name
            tool_args = json.loads(tc.function.arguments)

            # ★ query 교체: search_structured만. search_semantic은 GPT 추출값 사용 (임베딩 노이즈 방지)
            if "query" in tool_args and tool_name == "search_structured":
                original_query = tool_args["query"]
                tool_args["query"] = msg
                if original_query != msg:
                    log_lines.append(f"  [query 교체] \"{original_query[:30]}\" → 원문")
            elif "query" in tool_args and tool_name == "search_semantic":
                log_lines.append(f"  [query 유지] \"{tool_args['query'][:40]}\" (벡터 임베딩용)")

            log_lines.append(f"  [{tool_name}] {json.dumps(tool_args, ensure_ascii=False)[:80]}")

            if tool_name in ("search_structured", "search_semantic"):
                is_new_search = True

            # category 보완
            if "category" in tool_args and not tool_args.get("category"):
                if session_state.get("category"):
                    tool_args["category"] = session_state["category"]
                    log_lines.append(f"  [category 보완] {session_state['category']}")

            tool_fn = TOOL_MAP.get(tool_name)
            if not tool_fn:
                tool_results.append((tc, "error", f"알 수 없는 tool: {tool_name}", []))
                continue

            try:
                t0 = time.time()
                result_type, data, vendors = tool_fn(**tool_args)
                exec_time = time.time() - t0

                if result_type == "raw":
                    try:
                        records = json.loads(data)
                        log_lines.append(f"  [Neo4j] {len(records)}건 조회, {exec_time:.1f}초")
                    except:
                        log_lines.append(f"  [실행] {result_type}, {exec_time:.1f}초")
                else:
                    log_lines.append(f"  [실행] {result_type}, {exec_time:.1f}초")

                if vendors:
                    all_vendors.extend(vendors)
                    log_lines.append(f"  [vendors] {vendors[:3]}")

                tool_results.append((tc, result_type, data, vendors))
            except Exception as e:
                log_lines.append(f"  [오류] {type(e).__name__}: {e}")
                tool_results.append((tc, "error", str(e), []))

        # 결과 처리
        if num_calls == 1:
            tc, result_type, data, vendors = tool_results[0]

            if result_type == "direct":
                answer = data
            elif result_type == "graphrag":
                answer = data
            elif result_type == "raw":
                log_lines.append(f"[raw→LLM] {len(data)}자 → 답변 생성")
                messages.append(choice.message)
                messages.append({"role": "tool", "content": data, "tool_call_id": tc.id})
                try:
                    t0 = time.time()
                    final = client.chat.completions.create(
                        model="gpt-4o", messages=messages, temperature=0,
                    )
                    log_lines.append(f"[답변 생성] {time.time()-t0:.1f}초")
                    if final.usage:
                        log_lines.append(f"[tokens2] prompt={final.usage.prompt_tokens}, "
                                         f"completion={final.usage.completion_tokens}")
                    answer = final.choices[0].message.content
                except Exception as e2:
                    log_lines.append(f"[답변 생성 오류] {e2}")
                    answer = "죄송합니다. 답변 생성 중 오류가 발생했습니다."
            else:
                answer = "죄송합니다. 처리 중 오류가 발생했습니다."
        else:
            log_lines.append(f"[복수 tool] {num_calls}개 결과 합산")
            messages.append(choice.message)
            for tc, result_type, data, vendors in tool_results:
                content = data if result_type != "error" else f"오류: {data}"
                messages.append({"role": "tool", "content": content, "tool_call_id": tc.id})

            try:
                t0 = time.time()
                final = client.chat.completions.create(
                    model="gpt-4o", messages=messages, temperature=0,
                )
                log_lines.append(f"[답변 생성] {time.time()-t0:.1f}초")
                answer = final.choices[0].message.content
            except Exception as e2:
                log_lines.append(f"[답변 생성 오류] {e2}")
                answer = "죄송합니다. 답변 생성 중 오류가 발생했습니다."

    else:
        answer = choice.message.content or "웨딩 스드메(스튜디오/드레스/메이크업) 관련 질문을 해주세요."
        log_lines.append(f"[직접 답변] ({tool_select_time:.1f}초)")

    # ── 3단계: session_state 업데이트 ──
    if not all_vendors and choice.message.tool_calls:
        for tc in choice.message.tool_calls:
            args = json.loads(tc.function.arguments)
            if "vendor_names" in args:
                all_vendors = args["vendor_names"]
                break

    if all_vendors:
        seen = set()
        unique_vendors = []
        for v in all_vendors:
            if v not in seen:
                seen.add(v)
                unique_vendors.append(v)
        all_vendors = unique_vendors

        session_state["turn"] += 1
        session_state["last_mentioned"] = all_vendors
        if is_new_search:
            session_state["vendors"] = all_vendors
            if choice.message.tool_calls:
                for tc in choice.message.tool_calls:
                    cat = json.loads(tc.function.arguments).get("category")
                    if cat:
                        session_state["category"] = cat
                        session_state["vendor_history"][cat] = all_vendors
                        break
        log_lines.append(f"[저장] vendors={session_state['vendors'][:3]}, "
                         f"last={session_state['last_mentioned'][:3]}")
    else:
        log_lines.append(f"[저장] 변경 없음")

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": answer})

    new_log = "\n".join(log_lines)
    full_log = f"{debug_log}\n{'─'*40}\n{new_log}" if debug_log else new_log
    return "", chat_history, full_log


with gr.Blocks(title="웨딩 스드메 추천 챗봇", css="footer {display:none}") as demo:
    gr.Markdown("# 웨딩 스드메 추천 챗봇\n스튜디오 / 드레스 / 메이크업 추천을 한 곳에서! (서울/경기)")

    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### 처리 흐름")
            debug_box = gr.Textbox(label="", lines=30, max_lines=50, interactive=False)
            clear_debug_btn = gr.Button("로그 초기화", size="sm")
            clear_debug_btn.click(lambda: "", outputs=[debug_box])

        with gr.Column(scale=7):
            chatbot = gr.Chatbot(height=500)
            with gr.Row():
                msg = gr.Textbox(placeholder="예: 200만원 이하 야외씬 스튜디오 추천해줘", show_label=False, scale=9)
                send_btn = gr.Button("전송", scale=1)
            gr.Markdown("### 이런 질문을 해보세요")
            with gr.Row():
                gr.Button("스튜디오 추천").click(lambda: ("스튜디오 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("드레스 추천").click(lambda: ("드레스 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("메이크업 추천").click(lambda: ("메이크업 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("야외씬 잘 찍는곳").click(lambda: ("야외씬 잘 찍는곳 추천해줘", None), outputs=[msg, chatbot])
            with gr.Row():
                gr.Button("촬영+본식 드레스").click(lambda: ("촬영+본식 드레스 4벌 이상 추천", None), outputs=[msg, chatbot])
                gr.Button("내추럴 메이크업").click(lambda: ("내추럴 메이크업 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("리뷰 좋은 곳").click(lambda: ("리뷰 좋은 곳 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("내 취향 보기").click(lambda: ("내 취향 알려줘", None), outputs=[msg, chatbot])

    msg.submit(response_fn, [msg, chatbot, debug_box], [msg, chatbot, debug_box])
    send_btn.click(response_fn, [msg, chatbot, debug_box], [msg, chatbot, debug_box])

demo.launch(share=True)